# Crop

Wrap the `crop` step as a Common Workflow Language CommandLineTool and execute it with a CWL runner.

This notebook is linked to: https://eoap.github.io/mastering-app-package/cwl-cli/norm-diff/



The local CWL documents use imported EOAP types. Supply a YAML job document: STAC references are URI records with a `value` field; AOIs are Polygon records with `type`, `coordinates`, and `bbox`. The crop binding serializes the Polygon, and pixels outside it are masked. Workflow EPSG values are `"4326"`, and band values are `green`, `nir`, or `nir08`. The standalone crop tool still accepts a string EPSG code. Staged acquisitions remain CWL Directory objects. Published release examples elsewhere retain their original input format.


## Setup

In [ ]:
export WORKSPACE=/workspace/mastering-app-package
export RUNTIME=${WORKSPACE}/runs
mkdir -p ${RUNTIME}
rm -fr ${RUNTIME}/*
cd ${RUNTIME}


## Run the crop

Inspect and use `cwltool` to run the CommandLineTool definition:

The CWL document below shows the `crop` step wrapped as a CWL CommandLineTool:

In [ ]:

cat ${WORKSPACE}/cwl-cli/crop.cwl | yq e -

Run the CWL description for the `crop` command line tool:

In [ ]:
cat <<'EOF' > crop-green-params.yaml
item:
  value: 
    https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A
aoi:
  type: Polygon
  coordinates:
    -   -   - -121.399
            - 39.834
        -   - -120.74
            - 39.834
        -   - -120.74
            - 40.472
        -   - -121.399
            - 40.472
        -   - -121.399
            - 39.834
  bbox:
    - -121.399
    - 39.834
    - -120.74
    - 40.472
epsg: EPSG:4326
band: green
EOF

cwltool \
    --podman \
    --outdir ${WORKSPACE}/runs \
    ${WORKSPACE}/cwl-cli/crop.cwl \
    crop-green-params.yaml 2> crop.log


Let's look at the content of the stderr:

In [ ]:
cat crop.log

Remember the `cwltool` CWL runner can also take the parameters as a YAML file:


In [ ]:
cat <<'EOF' > params.yaml
item:
  value: 
    https://earth-search.aws.element84.com/v0/collections/sentinel-s2-l2a-cogs/items/S2B_10TFK_20210713_0_L2A
aoi:
  type: Polygon
  coordinates:
    -   -   - -121.399
            - 39.834
        -   - -120.74
            - 39.834
        -   - -120.74
            - 40.472
        -   - -121.399
            - 40.472
        -   - -121.399
            - 39.834
  bbox:
    - -121.399
    - 39.834
    - -120.74
    - 40.472
epsg: EPSG:4326
band: nir
EOF


Invoke `crop` against the NIR band redirecting the stdout to a file:

In [ ]:
cwltool \
    --podman \
    --outdir ${WORKSPACE}/runs \
    ${WORKSPACE}/cwl-cli/crop.cwl \
    params.yaml > results.json 

Let's inspect the stdout produced. The `output` block with the id `cropped` of the CWL description is a file:

In [ ]:
cat results.json

In [ ]:
cat ${WORKSPACE}/cwl-cli/crop.cwl | yq e .outputs -

Let's list the results produced:

In [ ]:
ls -l ${RUNTIME}